# AfriTune — Fon NER Baseline (XLM-RoBERTa)

**Public reproducibility edition — October 2026**

This notebook adapts the original single-run Google Colab experiment to an English-language research workflow. All historical metrics are reported in the repository scientific report; code outputs are intentionally cleared. Run the notebook in order on a Colab GPU to attempt reproduction; the dependency environment and results have not yet been verified. The dataset download now uses a pinned upstream reference snapshot, not a confirmed historical revision. Reproduction has not yet been independently validated.

**Data:** Fon subset of MasakhaNER 2.0. **Task:** BIO NER (PER, LOC, ORG, DATE). **Seed:** 42. **License:** Verify upstream dataset and model conditions before redistributing assets.


In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("GPU available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))
    print(
        "Mémoire GPU :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("Aucun GPU détecté")


In [ ]:
%pip install -q \
    "transformers>=4.55,<5" \
    "datasets>=3,<5" \
    "accelerate>=1,<2" \
    "seqeval>=1.2.2,<2" \
    "scikit-learn>=1.5,<2" \
    "sentencepiece>=0.2,<1" \
    "evaluate>=0.4,<1" \
    "requests>=2.31,<3" \
    "pandas>=2,<3" \
    "matplotlib>=3.8,<4"


In [ ]:
import torch
import transformers
import datasets
import accelerate
import seqeval
import sentencepiece
import sklearn

print("=== AFRITUNE ENVIRONMENT CHECK ===")
print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("Datasets      :", datasets.__version__)
print("Accelerate    :", accelerate.__version__)
print("Scikit-learn  :", sklearn.__version__)
print("Seqeval       : OK")
print("SentencePiece : OK")
print("GPU           :", torch.cuda.get_device_name(0))
print("=================================")
print("AfriTune environment ready!")


In [ ]:
# Record resolved package versions for reproducibility audits.
# This does not guarantee equivalence with the original training runtime.
import importlib.metadata as metadata
import json
import platform
from pathlib import Path

packages = [
    "torch", "transformers", "datasets", "accelerate",
    "evaluate", "seqeval", "numpy", "scikit-learn",
    "sentencepiece", "pandas", "matplotlib", "requests",
]
environment = {
    "python": platform.python_version(),
    "packages": {},
}
for package in packages:
    try:
        environment["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment["packages"][package] = "NOT INSTALLED"
print(json.dumps(environment, indent=2))
Path("afritune_environment.json").write_text(
    json.dumps(environment, indent=2) + "\n", encoding="utf-8"
)
print("Environment manifest written to afritune_environment.json")


In [ ]:
# Download a fixed upstream snapshot and verify the original Git blob hashes.
# This pinned snapshot is a reference for new runs; its identity with the
# historical training dataset has not yet been established.
import hashlib
import requests

DATASET_COMMIT = "ba5843cd08aa491d5f96a5e809e71eb9ec461391"
DATASET_BLOB_SHA1 = {
    "train": "e78ad5959205703a3b574cda7be8c4328475aa31",
    "dev": "c76280197f4d062b1ff72d37f5be32db0ab871d1",
    "test": "fbc739de7761f2a5983318649466718944340254",
}
base_url = (
    "https://raw.githubusercontent.com/"
    f"masakhane-io/masakhane-ner/{DATASET_COMMIT}/"
    "MasakhaNER2.0/data/fon"
)

fon_data = {}
for split, expected_blob in DATASET_BLOB_SHA1.items():
    response = requests.get(f"{base_url}/{split}.txt", timeout=60)
    response.raise_for_status()
    raw = response.content
    blob = hashlib.sha1(f"blob {len(raw)}\0".encode("ascii") + raw).hexdigest()
    if blob != expected_blob:
        raise ValueError(f"Dataset integrity check failed for {split}.txt")
    fon_data[split] = raw.decode("utf-8")
    print(f"{split}: verified ({len(raw):,} bytes); sha256={hashlib.sha256(raw).hexdigest()}")

print("Pinned Fon dataset snapshot downloaded and verified.")


In [ ]:
print("=== EXTRAIT DU DATASET FON ===\n")

print(fon_data["train"][:2000])


## Dataset analysis

Inspect sentence counts, word-token counts, and label frequencies before training.


In [ ]:
from collections import Counter
import pandas as pd

def analyze_fon_dataset(text):
    sentences = []
    current_sentence = []
    labels = Counter()

    for line in text.splitlines():
        line = line.strip()

        if not line:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
            continue

        if line.startswith("#"):
            continue

        parts = line.split()

        if len(parts) >= 2:
            token = parts[0]
            label = parts[-1]

            current_sentence.append((token, label))
            labels[label] += 1

    if current_sentence:
        sentences.append(current_sentence)

    return sentences, labels


results = []

for split in ["train", "dev", "test"]:
    sentences, labels = analyze_fon_dataset(
        fon_data[split]
    )

    results.append({
        "Split": split,
        "Phrases": len(sentences),
        "Tokens": sum(len(s) for s in sentences),
        "Labels uniques": len(labels)
    })

    print(f"\n=== {split.upper()} ===")
    print("Distribution des labels :")
    print(dict(labels))

df = pd.DataFrame(results)

print("\n=== RÉSUMÉ DATASET FON ===")
display(df)


## NER labels and corpus overview

The corpus uses BIO annotations for PER, LOC, ORG, and DATE. The outside label is O.


## Build Hugging Face datasets

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
from datasets import Dataset, DatasetDict

def parse_ner_data(text):
    examples = []
    tokens = []
    tags = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            if tokens:
                examples.append({
                    "tokens": tokens,
                    "ner_tags": tags
                })
                tokens = []
                tags = []
            continue

        if line.startswith("#"):
            continue

        parts = line.split()

        if len(parts) < 2:
            raise ValueError(f"Invalid line : {line}")

        tokens.append(parts[0])
        tags.append(parts[-1])

    if tokens:
        examples.append({
            "tokens": tokens,
            "ner_tags": tags
        })

    return examples


dataset = DatasetDict({
    "train": Dataset.from_list(
        parse_ner_data(fon_data["train"])
    ),
    "validation": Dataset.from_list(
        parse_ner_data(fon_data["dev"])
    ),
    "test": Dataset.from_list(
        parse_ner_data(fon_data["test"])
    )
})

print("=== AFRITUNE DATASET ===")
print(dataset)

print("\n=== FIRST SENTENCE ===")
print(dataset["train"][0])


## XLM-RoBERTa tokenization

The tokenizer can split Fon words into several subtokens. Labels must be aligned with word IDs.


## Load the tokenizer

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer loaded successfully !")
print("Modèle :", MODEL_NAME)
print("Type :", type(tokenizer).__name__)

example = dataset["train"][0]

encoded = tokenizer(
    example["tokens"],
    is_split_into_words=True,
    truncation=True,
    max_length=128
)

print("\n=== ORIGINAL WORDS ===")
print(example["tokens"][:10])

print("\n=== SUBTOKENS XLM-R ===")
print(tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][:30]
))

print("\n=== ALIGNMENT WORD IDs ===")
print(encoded.word_ids()[:30])


## Align BIO annotations to subtokens

Only the first subtoken of each word receives its BIO label; subsequent subtokens and special tokens use -100.


## Create label IDs

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
label_list = sorted({
    tag
    for example in dataset["train"]
    for tag in example["ner_tags"]
})

# Placer O en premier
label_list = ["O"] + [
    label for label in label_list if label != "O"
]

label2id = {
    label: i for i, label in enumerate(label_list)
}

id2label = {
    i: label for label, i in label2id.items()
}

print("=== ÉTIQUETTES AFRITUNE ===")
print(label2id)
print("Nombre de classes :", len(label_list))


## Apply label alignment

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    all_labels = []

    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)

        previous_word_id = None
        aligned = []

        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)

            elif word_id != previous_word_id:
                aligned.append(
                    label2id[tags[word_id]]
                )

            else:
                aligned.append(-100)

            previous_word_id = word_id

        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("=== TOKENIZED DATASET ===")
print(tokenized_dataset)


In [ ]:
sample = tokenized_dataset["train"][0]

tokens = tokenizer.convert_ids_to_tokens(
    sample["input_ids"]
)

print("=== ALIGNMENT TOKENS / LABELS ===")

for token, label_id in zip(
    tokens[:35],
    sample["labels"][:35]
):
    label = (
        "IGNORED"
        if label_id == -100
        else id2label[label_id]
    )

    print(f"{token:15} -> {label}")

assert len(sample["input_ids"]) == len(sample["labels"])

print("\nAlignement validé !")


## Audit truncation

Follow the next cell to reproduce this stage of the experiment.


## Why 128 subtokens may be insufficient

A maximum length of 128 may truncate Fon sentences because of subword fragmentation. The following cells audit this behavior.


In [ ]:
import numpy as np

print("=== AFRITUNE — TOKENIZATION AUDIT ===")

for split in ["train", "validation", "test"]:

    original = dataset[split]
    tokenized = tokenized_dataset[split]

    truncated_sentences = 0
    total_words = 0
    lost_words = 0
    lost_entities = 0
    total_entities_tokens = 0
    token_lengths = []

    for example in original:

        full_encoding = tokenizer(
            example["tokens"],
            is_split_into_words=True,
            truncation=False
        )

        word_ids = full_encoding.word_ids()

        token_lengths.append(
            len(full_encoding["input_ids"])
        )

        if len(full_encoding["input_ids"]) > 128:
            truncated_sentences += 1

        kept_encoding = tokenizer(
            example["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=128
        )

        kept_word_ids = {
            idx for idx in kept_encoding.word_ids()
            if idx is not None
        }

        total_words += len(example["tokens"])

        for i, label in enumerate(example["ner_tags"]):

            if label != "O":
                total_entities_tokens += 1

            if i not in kept_word_ids:
                lost_words += 1

                if label != "O":
                    lost_entities += 1

    print(f"\n--- {split.upper()} ---")
    print("Phrases truncated :", truncated_sentences)
    print(
        "Taux de sentences truncated :",
        round(100 * truncated_sentences / len(original), 2),
        "%"
    )
    print(
        "Longueur moyenne en sous-tokens :",
        round(np.mean(token_lengths), 2)
    )
    print(
        "Longueur P95 :",
        round(np.percentile(token_lengths, 95), 2)
    )
    print("Lost words :", lost_words)
    print("Tokens d'entités lost :", lost_entities)
    print(
        "Taux de tokens d'entités lost :",
        round(
            100 * lost_entities /
            max(total_entities_tokens, 1),
            2
        ),
        "%"
    )


In [ ]:
import matplotlib.pyplot as plt

splits = ["Train", "Validation", "Test"]
truncation_rates = [12.80, 9.31, 9.93]

plt.figure(figsize=(9, 5))

bars = plt.bar(
    splits,
    truncation_rates,
    color=["#2563EB", "#10B981", "#F59E0B"],
    width=0.55
)

for bar, value in zip(bars, truncation_rates):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.3,
        f"{value:.2f} %",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

plt.title(
    "AfriTune — Impact de la troncature sur le dataset Fon",
    fontsize=14,
    fontweight="bold"
)
plt.ylabel("Truncated sentences (%)")
plt.xlabel("Partition du dataset")
plt.ylim(0, 16)
plt.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.savefig(
    "afritune_truncation_128.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## Observed truncation at 128 subtokens

The original 128-subtoken audit reported truncation rates of 12.80% (train), 9.31% (validation), and 9.93% (test). These are historical observations, not newly rerun measurements.


## Compare 128 and 256 subtokens

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import pandas as pd

results = []

for split in ["train", "validation", "test"]:

    for max_length in [128, 256]:

        truncated = 0
        lost_words = 0
        lost_entity_tokens = 0
        total_entity_tokens = 0

        for example in dataset[split]:

            full = tokenizer(
                example["tokens"],
                is_split_into_words=True,
                truncation=False
            )

            limited = tokenizer(
                example["tokens"],
                is_split_into_words=True,
                truncation=True,
                max_length=max_length
            )

            if len(full["input_ids"]) > max_length:
                truncated += 1

            kept_ids = {
                i for i in limited.word_ids()
                if i is not None
            }

            for i, label in enumerate(example["ner_tags"]):

                if label != "O":
                    total_entity_tokens += 1

                if i not in kept_ids:
                    lost_words += 1

                    if label != "O":
                        lost_entity_tokens += 1

        results.append({
            "Split": split,
            "Max length": max_length,
            "Truncated sentences (%)": round(
                100 * truncated / len(dataset[split]), 2
            ),
            "Lost words": lost_words,
            "Lost entity tokens": lost_entity_tokens,
            "Entity token loss (%)": round(
                100 * lost_entity_tokens /
                max(total_entity_tokens, 1), 2
            )
        })

comparison_df = pd.DataFrame(results)

print("=== AFRITUNE : COMPARAISON 128 VS 256 ===")
display(comparison_df)


## Select 256 subtokens

The original comparison found no truncated sentences at a maximum length of 256 on the three splits. The final training pipeline therefore uses 256.


## Rebuild the tokenized dataset

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
MAX_LENGTH = 256

def tokenize_and_align_labels_256(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    all_labels = []

    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        aligned = []

        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != previous_word_id:
                aligned.append(label2id[tags[word_id]])
            else:
                aligned.append(-100)

            previous_word_id = word_id

        all_labels.append(aligned)

    tokenized["labels"] = all_labels
    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align_labels_256,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("=== AFRITUNE — DATASET FINAL ===")
print(tokenized_dataset)

for split in ["train", "validation", "test"]:
    max_observed = max(
        len(row["input_ids"])
        for row in tokenized_dataset[split]
    )

    alignment_ok = all(
        len(row["input_ids"]) == len(row["labels"])
        for row in tokenized_dataset[split]
    )

    print(f"\n{split.upper()}")
    print("Sentence count :", len(tokenized_dataset[split]))
    print("Maximum observed length :", max_observed)
    print("Valid alignment :", alignment_ok)

    assert max_observed <= MAX_LENGTH
    assert alignment_ok

print("\nDataset ready for model configuration !")


## Pipeline architecture

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

steps = [
    ("Corpus Fon — MasakhaNER 2.0",
     "Train : 4343 | Validation : 623 | Test : 1228"),
    ("Tokenisation XLM-R",
     "SentencePiece | max_length = 256"),
    ("Alignement BIO",
     "9 labels | sous-tokens ignorés : -100"),
    ("XLM-RoBERTa Base",
     "Encodeur Transformer multilingue"),
    ("Classification NER",
     "O | PER | LOC | ORG | DATE"),
    ("Évaluation",
     "Précision | Rappel | F1 par entité")
]

fig, ax = plt.subplots(figsize=(11, 11))
ax.set_xlim(0, 10)
ax.set_ylim(0, 13)
ax.axis("off")

colors = [
    "#E8EDF5",
    "#DBEAFE",
    "#E8EDF5",
    "#BFDBFE",
    "#D1FAE5",
    "#E8EDF5"
]

for i, ((title, subtitle), color) in enumerate(
    zip(steps, colors)
):
    y = 11.2 - i * 2

    box = FancyBboxPatch(
        (1.3, y),
        7.4,
        1.3,
        boxstyle="round,pad=0.12",
        facecolor=color,
        edgecolor="#334155",
        linewidth=1.3
    )
    ax.add_patch(box)

    ax.text(
        5, y + 0.85, title,
        ha="center", va="center",
        fontsize=13, fontweight="bold",
        color="#0F172A"
    )

    ax.text(
        5, y + 0.38, subtitle,
        ha="center", va="center",
        fontsize=10, color="#475569"
    )

    if i < len(steps) - 1:
        arrow = FancyArrowPatch(
            (5, y - 0.12),
            (5, y - 0.60),
            arrowstyle="-|>",
            mutation_scale=18,
            linewidth=1.5,
            color="#475569"
        )
        ax.add_patch(arrow)

ax.set_title(
    "AfriTune Architecture du pipeline NER Fon",
    fontsize=17,
    fontweight="bold",
    pad=20
)

plt.savefig(
    "afritune_architecture.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "afritune_architecture.pdf",
    bbox_inches="tight"
)

plt.show()


## Training configuration

The experiment uses three epochs, learning rate 2e-5, batch size 4, gradient accumulation 4, FP16 on GPU, and seed 42.


In [ ]:
import os
import random
import numpy as np
import torch
import evaluate

from transformers import (
    TrainingArguments,
    DataCollatorForTokenClassification,
    set_seed
)

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

EXPERIMENT_NAME = "AfriTune-Fon-XLMR-Base-v1"
OUTPUT_DIR = f"./results/{EXPERIMENT_NAME}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Padding dynamique
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_tags = []
        true_tags = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                pred_tags.append(id2label[int(pred_id)])
                true_tags.append(id2label[int(label_id)])

        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none"
)

print("=== AFRITUNE — TRAINING CONFIGURATION ===")
print("Experiment :", EXPERIMENT_NAME)
print("Epochs :", training_args.num_train_epochs)
print("Learning rate :", training_args.learning_rate)
print("Batch size :", training_args.per_device_train_batch_size)
print("Accumulation :", training_args.gradient_accumulation_steps)
print("FP16 :", training_args.fp16)
print("Seed :", training_args.seed)
print("Primary metric : F1 NER")
print("\nConfiguration ready !")


## Initialize the Trainer

Follow the next cell to reproduce this stage of the experiment.


## Load the model

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import torch
from transformers import AutoModelForTokenClassification

MODEL_NAME = "FacebookAI/xlm-roberta-base"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("=== AFRITUNE — MODEL LOADED ===")
print("Architecture :", model.config.model_type)
print("Nombre de classes :", model.config.num_labels)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Paramètres : {total_params:,}")
print("GPU available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("\nModèle loaded avec success !")


In [ ]:
from transformers import Trainer

assert model.config.num_labels == 9
assert len(tokenized_dataset["train"]) == 4343
assert len(tokenized_dataset["validation"]) == 623

# Initialisation du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("=== AFRITUNE — TRAINER INITIALISÉ ===")
print("Modèle :", model.config.model_type)
print("Train :", len(trainer.train_dataset), "sentences")
print("Validation :", len(trainer.eval_dataset), "sentences")
print("Epochs :", trainer.args.num_train_epochs)
print("FP16 :", trainer.args.fp16)
print("Primary metric :", trainer.args.metric_for_best_model)

print("\nTrainer prêt pour l'training !")


## Fine-tuning on a Tesla T4

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import torch

print("=== AFRITUNE — FIRST FINE-TUNING ===")

assert torch.cuda.is_available(), "GPU CUDA non disponible"

print("GPU :", torch.cuda.get_device_name(0))
print("Experiment :", EXPERIMENT_NAME)

train_result = trainer.train()

print("\n=== TRAINING COMPLETED ===")
print("Training loss :", train_result.training_loss)

trainer.save_model(
    f"{OUTPUT_DIR}/best_model"
)

tokenizer.save_pretrained(
    f"{OUTPUT_DIR}/best_model"
)

trainer.save_metrics(
    "train",
    train_result.metrics
)

trainer.save_state()

print("\nModel saved to :")
print(f"{OUTPUT_DIR}/best_model")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

history = pd.DataFrame(trainer.state.log_history)

eval_history = history[
    history["eval_f1"].notna()
].copy()

FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.figure(figsize=(9, 5))

for metric, label in [
    ("eval_f1", "F1"),
    ("eval_precision", "Précision"),
    ("eval_recall", "Rappel")
]:
    plt.plot(
        eval_history["epoch"],
        eval_history[metric],
        marker="o",
        linewidth=2,
        label=label
    )

plt.title("AfriTune — Performances NER sur le Fon")
plt.xlabel("Époque")
plt.ylabel("Score")
plt.xticks([1, 2, 3])
plt.ylim(0.70, 1.0)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES_DIR, "ner_metrics.png"),
    dpi=300
)
plt.show()

# Figure 2 : Loss de validation
plt.figure(figsize=(9, 5))

plt.plot(
    eval_history["epoch"],
    eval_history["eval_loss"],
    marker="o",
    linewidth=2
)

plt.title("AfriTune — Évolution de la validation loss")
plt.xlabel("Époque")
plt.ylabel("Validation loss")
plt.xticks([1, 2, 3])
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES_DIR, "validation_loss.png"),
    dpi=300
)
plt.show()

print("=== FIGURES SAUVEGARDÉES ===")
print(FIGURES_DIR)

print("\n=== HISTORIQUE DES ÉVALUATIONS ===")
display(
    eval_history[
        ["epoch", "eval_loss",
         "eval_precision", "eval_recall", "eval_f1"]
    ]
)


## Validation learning curves

Validation F1 improved across the three training epochs in the original run. Results below are descriptive and come from one seed.


## Per-entity validation metrics

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import numpy as np
import pandas as pd
from seqeval.metrics import classification_report

print("=== AFRITUNE — PER-ENTITY EVALUATION ===")

predictions_output = trainer.predict(
    tokenized_dataset["validation"]
)

predictions = np.argmax(
    predictions_output.predictions,
    axis=2
)

labels = predictions_output.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):

    pred_tags = []
    true_tags = []

    for pred_id, label_id in zip(prediction, label):

        if label_id != -100:
            pred_tags.append(id2label[int(pred_id)])
            true_tags.append(id2label[int(label_id)])

    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

print("\n=== NER REPORT BY ENTITY TYPE ===")

report = classification_report(
    true_labels,
    true_predictions,
    digits=4
)

print(report)

# Rapport sous forme de tableau
report_dict = classification_report(
    true_labels,
    true_predictions,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T

display(report_df)

report_df.to_csv(
    f"{OUTPUT_DIR}/ner_validation_report.csv",
    index=True
)

print("\nReport saved successfully !")


## Validation interpretation

The original validation analysis reported strong performance on PER and LOC. These validation results must not be confused with the held-out test scores.


In [ ]:
import pandas as pd

errors = []

for i, (gold, pred) in enumerate(
    zip(true_labels, true_predictions)
):
    if gold != pred:
        errors.append({
            "sentence_id": i,
            "texte": " ".join(dataset["validation"][i]["tokens"]),
            "labels_reels": " ".join(gold),
            "labels_predits": " ".join(pred),
            "tokens_incorrects": sum(
                g != p for g, p in zip(gold, pred)
            )
        })

errors_df = pd.DataFrame(errors)

print("=== AFRITUNE — ANALYSE DES ERREURS ===")
print("Phrases de validation :", len(dataset["validation"]))
print("Phrases avec errors :", len(errors_df))

print(
    "Taux de sentences avec errors :",
    round(
        100 * len(errors_df) /
        len(dataset["validation"]), 2
    ),
    "%"
)

print("\n=== EXEMPLES D'ERREURS ===")

display(
    errors_df[
        ["sentence_id", "texte", "tokens_incorrects"]
    ].head(10)
)

errors_df.to_csv(
    f"{OUTPUT_DIR}/validation_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFichier sauvegardé : validation_errors.csv")


## Sentence-level validation errors

In the original validation analysis, all supervised labels matched for 524 of 623 sentences. Sentence-level exact match is distinct from entity-level F1.


## Detailed error analysis

Follow the next cell to reproduce this stage of the experiment.


## Error analysis objectives

Inspect each mismatched token, its gold BIO label, its predicted label, and the corresponding sentence ID.


In [ ]:
import pandas as pd
from collections import Counter

detailed_errors = []

for sentence_id, (gold, pred) in enumerate(
    zip(true_labels, true_predictions)
):
    original_tokens = dataset["validation"][sentence_id]["tokens"]

    assert len(original_tokens) == len(gold) == len(pred), (
        f"Alignement incorrect pour la sentence {sentence_id}"
    )

    for token_id, (token, true_tag, pred_tag) in enumerate(
        zip(original_tokens, gold, pred)
    ):
        if true_tag != pred_tag:
            detailed_errors.append({
                "sentence_id": sentence_id,
                "token_id": token_id,
                "token": token,
                "label_reel": true_tag,
                "label_predit": pred_tag,
                "type_erreur": f"{true_tag} → {pred_tag}"
            })

detailed_df = pd.DataFrame(detailed_errors)

print("=== AFRITUNE — ERREURS DÉTAILLÉES ===")
print("Nombre total d'errors de tokens :", len(detailed_df))

print("\n=== 20 PREMIÈRES ERREURS ===")
display(detailed_df.head(20))

print("\n=== CONFUSIONS LES PLUS FRÉQUENTES ===")
confusions = (
    detailed_df["type_erreur"]
    .value_counts()
    .head(15)
    .reset_index()
)

confusions.columns = ["Type de confusion", "Occurrences"]
display(confusions)

detailed_df.to_csv(
    f"{OUTPUT_DIR}/detailed_validation_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nAnalyse détaillée sauvegardée !")


## Error analysis interpretation

The original error analysis identified notable confusions involving ORG and DATE. Further analysis is needed to explain the causes.


## Final held-out test evaluation

Follow the next cell to reproduce this stage of the experiment.


## Test evaluation protocol

The held-out test split contains 1,228 sentences and is evaluated after selecting the best checkpoint on validation.


In [ ]:
import numpy as np
import pandas as pd
from seqeval.metrics import classification_report

print("=== AFRITUNE — FINAL TEST EVALUATION ===")

print("Best checkpoint :", trainer.state.best_model_checkpoint)
print("Best validation F1 :", trainer.state.best_metric)

test_output = trainer.predict(
    tokenized_dataset["test"]
)

print("\n=== OVERALL METRICS — TEST ===")
print(test_output.metrics)

pred_ids = np.argmax(
    test_output.predictions,
    axis=2
)

test_true_labels = []
test_pred_labels = []

for predictions, labels in zip(
    pred_ids,
    test_output.label_ids
):
    gold_tags = []
    predicted_tags = []

    for pred_id, label_id in zip(predictions, labels):
        if label_id != -100:
            gold_tags.append(id2label[int(label_id)])
            predicted_tags.append(id2label[int(pred_id)])

    test_true_labels.append(gold_tags)
    test_pred_labels.append(predicted_tags)

print("\n=== NER REPORT — TEST ===")

print(
    classification_report(
        test_true_labels,
        test_pred_labels,
        digits=4
    )
)

report_dict = classification_report(
    test_true_labels,
    test_pred_labels,
    output_dict=True
)

test_report_df = pd.DataFrame(report_dict).T

test_report_df.to_csv(
    f"{OUTPUT_DIR}/ner_test_report.csv",
    encoding="utf-8-sig"
)

print("\nTest report saved !")


## Validation versus test comparison

Follow the next cell to reproduce this stage of the experiment.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# Langue : Fon
# =========================================================

FIGURES_DIR = os.path.join(
    OUTPUT_DIR, "figures", "comparative_study"
)
os.makedirs(FIGURES_DIR, exist_ok=True)

global_metrics = pd.DataFrame({
    "Métrique": ["Précision", "Rappel", "F1"],
    "Validation": [
        0.837050, 0.876122, 0.856140
    ],
    "Test": [
        0.773958, 0.809487, 0.791324
    ]
})

entity_metrics = pd.DataFrame({
    "Entité": ["PER", "LOC", "ORG", "DATE"],
    "Validation": [
        0.915888, 0.900901, 0.805654, 0.812903
    ],
    "Test": [
        0.8840, 0.8568, 0.7385, 0.6771
    ]
})

global_metrics["Écart (pts)"] = (
    global_metrics["Test"]
    - global_metrics["Validation"]
) * 100

entity_metrics["Écart (pts)"] = (
    entity_metrics["Test"]
    - entity_metrics["Validation"]
) * 100

# ---------------------------------------------------------
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(global_metrics))
width = 0.35

bars1 = ax.bar(
    x - width / 2,
    global_metrics["Validation"] * 100,
    width,
    label="Validation",
    color="#2563EB"
)

bars2 = ax.bar(
    x + width / 2,
    global_metrics["Test"] * 100,
    width,
    label="Test",
    color="#F59E0B"
)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.2f}%",
            (
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=10
        )

ax.set_title(
    "AfriTune — Comparaison des performances globales",
    fontsize=15,
    fontweight="bold"
)

ax.set_ylabel("Score (%)")
ax.set_xticks(x)
ax.set_xticklabels(global_metrics["Métrique"])
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES_DIR, "global_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ---------------------------------------------------------
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(entity_metrics))

bars1 = ax.bar(
    x - width / 2,
    entity_metrics["Validation"] * 100,
    width,
    label="Validation",
    color="#2563EB"
)

bars2 = ax.bar(
    x + width / 2,
    entity_metrics["Test"] * 100,
    width,
    label="Test",
    color="#F59E0B"
)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.2f}%",
            (
                bar.get_x() + bar.get_width() / 2,
                height
            ),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=9
        )

ax.set_title(
    "AfriTune — F1 par category d'entité",
    fontsize=15,
    fontweight="bold"
)

ax.set_ylabel("F1-score (%)")
ax.set_xticks(x)
ax.set_xticklabels(entity_metrics["Entité"])
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES_DIR, "entity_f1_comparison.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ---------------------------------------------------------
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

gap = entity_metrics["Écart (pts)"]

bars = ax.barh(
    entity_metrics["Entité"],
    gap,
    color="#DC2626"
)

for bar, value in zip(bars, gap):
    ax.text(
        value - 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.2f} pts",
        va="center",
        ha="right",
        fontsize=10
    )

ax.axvline(0, color="black", linewidth=1)

ax.set_title(
    "AfriTune — Écart de généralisation par entité",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel(
    "Différence F1 : Test - Validation (points)"
)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()

plt.savefig(
    os.path.join(FIGURES_DIR, "generalization_gap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ---------------------------------------------------------
# TABLEAUX COMPARATIFS
# ---------------------------------------------------------

print("\n=== OVERALL METRICS ===")
display(global_metrics.round(4))

print("\n=== F1 PAR ENTITÉ ===")
display(entity_metrics.round(4))

# Export CSV
global_metrics.to_csv(
    os.path.join(FIGURES_DIR, "global_metrics.csv"),
    index=False
)

entity_metrics.to_csv(
    os.path.join(FIGURES_DIR, "entity_metrics.csv"),
    index=False
)

print("\n=== INTERPRÉTATION AUTOMATIQUE ===")

global_gap = global_metrics.loc[
    global_metrics["Métrique"] == "F1",
    "Écart (pts)"
].iloc[0]

worst = entity_metrics.loc[
    entity_metrics["Écart (pts)"].idxmin()
]

best = entity_metrics.loc[
    entity_metrics["Test"].idxmax()
]

print(
    f"Écart F1 global : {global_gap:.2f} points"
)
print(
    f"Plus forte baisse : {worst['Entité']} "
    f"({worst['Écart (pts)']:.2f} points)"
)
print(
    f"Meilleure category sur test : "
    f"{best['Entité']} ({best['Test']*100:.2f} %)"
)

print("\nFigures sauvegardées dans :", FIGURES_DIR)


## Comparative results and limitations

The original run obtained validation entity F1 0.856140 and held-out test entity F1 0.791324. This is one run, not a confidence interval or state-of-the-art comparison.


## Optional Google Drive backup

Backup is optional and should be performed only after checking available Google Drive storage. No files are deleted by this notebook.


In [ ]:
from google.colab import drive
import os
import json
import shutil
import torch
import transformers
import datasets
import numpy as np

# 1. Mount Google Drive
drive.mount("/content/drive")

EXPERIMENT_NAME = "AfriTune-Fon-XLMR-Base-v1"

SOURCE_DIR = os.path.abspath(
    f"./results/{EXPERIMENT_NAME}"
)

DRIVE_DIR = (
    os.path.join(os.environ.get("AFRITUNE_DRIVE_ROOT", "/content/drive/MyDrive/AfriTune/experiments"), EXPERIMENT_NAME)
)

assert os.path.isdir(SOURCE_DIR), (
    f"Directory not found : {SOURCE_DIR}"
)

assert os.path.isdir(
    os.path.join(SOURCE_DIR, "best_model")
), "Saved model not found."

os.makedirs(os.path.dirname(DRIVE_DIR), exist_ok=True)

shutil.copytree(
    SOURCE_DIR,
    DRIVE_DIR,
    dirs_exist_ok=True
)

metadata = {
    "experiment": EXPERIMENT_NAME,
    "model": "FacebookAI/xlm-roberta-base",
    "dataset": "MasakhaNER 2.0",
    "language": "fon",
    "task": "Named Entity Recognition",
    "labels": label_list,
    "train_sentences": 4343,
    "validation_sentences": 623,
    "test_sentences": 1228,
    "seed": 42,
    "max_length": 256,
    "epochs": 3,
    "learning_rate": 2e-5,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "fp16": True,
    "best_validation_f1": 0.8561403508771931,
    "test_precision": 0.7739575713240673,
    "test_recall": 0.809487375669472,
    "test_f1": 0.7913238593866867,
    "test_accuracy": 0.9820383949905248,
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "numpy": np.__version__
    }
}

metadata_path = os.path.join(
    DRIVE_DIR, "experiment_metadata.json"
)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False
    )

print("=== AFRITUNE — SAUVEGARDE ===")
print("Source:", SOURCE_DIR)
print("Destination:", DRIVE_DIR)

print("\nSaved files :")
for item in sorted(os.listdir(DRIVE_DIR)):
    print(" -", item)

print("\nMetadata :", os.path.isfile(metadata_path))
print("\nBackup completed successfully !")


In [ ]:
import pandas as pd
import os

results = pd.DataFrame({
    "Entity": ["DATE", "LOC", "ORG", "PER", "Micro average"],
    "Precision": [0.6557, 0.8214, 0.7467, 0.8796, 0.7740],
    "Recall": [0.7000, 0.8954, 0.7304, 0.8885, 0.8095],
    "F1-score": [0.6771, 0.8568, 0.7385, 0.8840, 0.7913],
    "Support": [370, 411, 230, 296, 1307]
})

print("=== AFRITUNE — TEST RESULTS ===")
display(results)

# Export CSV
results.to_csv(
    f"{OUTPUT_DIR}/afritune_baseline_test_results.csv",
    index=False
)

latex_table = results.to_latex(
    index=False,
    float_format="%.4f",
    caption="NER performance of XLM-RoBERTa Base on Fon (MasakhaNER 2.0)",
    label="tab:afritune_fon_baseline"
)

latex_path = f"{OUTPUT_DIR}/afritune_baseline_test_results.tex"

with open(latex_path, "w", encoding="utf-8") as f:
    f.write(latex_table)

print("\nTableau scientifique exporté :")
print(latex_path)
